<a href="https://colab.research.google.com/github/devarsh3453/NEW/blob/main/YouTube_RAG_HuggingFace_Local.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# YouTube Transcript RAG Bot — Local Hugging Face LLM

This notebook retrieves a YouTube transcript, indexes it with local Hugging Face embeddings and FAISS, then answers questions using a locally running Qwen model.


- The small `Qwen/Qwen2.5-0.5B-Instruct` model is chosen to be more practical on CPU.

In [ ]:
%pip install -q -U youtube-transcript-api langchain-core langchain-community langchain-text-splitters langchain-huggingface sentence-transformers faiss-cpu transformers accelerate

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 485.2/485.2 kB 7.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 573.0/573.0 kB 18.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.4/2.4 MB 47.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 740.6/740.6 kB 14.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 18.8/18.8 MB 54.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 12.6/12.6 MB 50.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.0/1.0 MB 36.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 69.4/69.4 kB 2.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 73.1/73.1 kB 2.9 MB/s eta 0:00:00
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
google-colab 1.0.0 requires requests==2.32.4, but you have requests 2.34.2 which is incompatible.


In [ ]:
import torch
from youtube_transcript_api import YouTubeTranscriptApi, TranscriptsDisabled
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_huggingface import HuggingFaceEmbeddings
from langchain_community.vectorstores import FAISS
from langchain_core.prompts import PromptTemplate
from transformers import AutoTokenizer, AutoModelForCausalLM

/tmp/ipykernel_740/3312148347.py:5: DeprecationWarning: `langchain-community` is being sunset and is no longer actively maintained. See https://github.com/langchain-ai/langchain-community/issues/674 for details and migration guidance toward standalone integration packages.
  from langchain_community.vectorstores import FAISS


## 1. Get the YouTube transcript
Change `video_id` to the ID of the video you want to query.

In [ ]:
video_id = "Gfr50f6ZBvo"  # Only the video ID, not the full URL

try:
    fetched_transcript = YouTubeTranscriptApi().fetch(
        video_id,
        languages=["en"]
    )
    transcript = " ".join(snippet.text for snippet in fetched_transcript)
    print(f"Transcript loaded: {len(transcript):,} characters")
except TranscriptsDisabled:
    raise RuntimeError("Captions are disabled for this video.")

Transcript loaded: 133,836 characters


## 2. Split transcript into chunks

In [ ]:
splitter = RecursiveCharacterTextSplitter(
    chunk_size=1000,
    chunk_overlap=200
)
chunks = splitter.create_documents([transcript])
print(f"Created {len(chunks)} chunks")

Created 168 chunks


## 3. Create embeddings, FAISS index, and MMR retriever

In [ ]:
embeddings = HuggingFaceEmbeddings(
    model_name="sentence-transformers/all-MiniLM-L6-v2"
)

vector_store = FAISS.from_documents(chunks, embeddings)

retriever = vector_store.as_retriever(
    search_type="mmr",
    search_kwargs={
        "k": 4,
        "fetch_k": 20,
        "lambda_mult": 0.5
    }
)

print("Vector store and retriever are ready.")

modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

README.md:   0%|          | 0.00/10.5k [00:00<?, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/612 [00:00<?, ?B/s]

...

vocab.txt:   0%|          | 0.00/232k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/466k [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

Vector store and retriever are ready.


## 4. Load a local instruction-tuned LLM
This model runs on CPU and does not call a hosted inference endpoint. The first run downloads the model weights.

In [ ]:
model_id = "Qwen/Qwen2.5-0.5B-Instruct"

tokenizer = AutoTokenizer.from_pretrained(model_id)
model = AutoModelForCausalLM.from_pretrained(
    model_id,
    torch_dtype=torch.float32
)
model.to("cpu")
model.eval()

print("Model loaded on CPU.")

config.json:   0%|          | 0.00/659 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/7.30k [00:00<?, ?B/s]

vocab.json:   0%|          | 0.00/2.78M [00:00<?, ?B/s]

merges.txt:   0%|          | 0.00/1.67M [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/7.03M [00:00<?, ?B/s]

...

model.safetensors: reconstructing file:   0%|          |  0.00B /  988MB            

model.safetensors: downloading bytes:           |  0.00B            

Loading weights:   0%|          | 0/290 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/242 [00:00<?, ?B/s]

Model loaded on CPU.


## 5. Ask questions
The answer is generated from the transcript chunks retrieved for the question. `temperature=0.1` is set in the generation call.

In [ ]:
prompt = PromptTemplate(
    template="""Answer using ONLY the YouTube transcript context below.
If the context does not contain enough information to answer, say "I don't know".

Transcript context:
{context}

Question: {question}

Answer:""",
    input_variables=["context", "question"]
)

def answer_question(question: str) -> str:
    retrieved_docs = retriever.invoke(question)
    context_text = "\n\n".join(doc.page_content for doc in retrieved_docs)
    final_prompt = prompt.format(context=context_text, question=question)

    messages = [
        {
            "role": "system",
            "content": "You are a helpful assistant. Ground every answer only in the provided transcript context."
        },
        {
            "role": "user",
            "content": final_prompt
        }
    ]

    inputs = tokenizer.apply_chat_template(
        messages,
        add_generation_prompt=True,
        tokenize=True,
        return_tensors="pt",
        return_dict=True
    )
    input_length = inputs["input_ids"].shape[-1]

    with torch.no_grad():
        outputs = model.generate(
            **inputs,
            max_new_tokens=200,
            do_sample=True,
            temperature=0.1,
            top_p=0.9,
            repetition_penalty=1.05
        )

    new_tokens = outputs[0][input_length:]
    return tokenizer.decode(new_tokens, skip_special_tokens=True).strip()

In [ ]:
question = "Is the topic of nuclear fusion discussed in this video? If yes, what was discussed?"
answer = answer_question(question)
print(answer)

Yes, the topic of nuclear fusion is discussed in this video. The video mentions that the Swiss Technical Institute (Swiss Technikzentrum) has a test reactor that they are willing to let the company use. They also mention that the Swiss Technikzentrum has a test reactor that they are willing to let the company use.


## Ask another question



In [ ]:
question = "Can you summarize the video?"
print(answer_question(question))

The video discusses the origins of Claude Shannon's chess-playing program, Kasparov, and its impact on the field of computer science. It highlights the importance of Kasparov's human-like intelligence in advancing scientific research, particularly in biology, where he has helped establish fundamental research into proteins. The video also mentions the use of this technology by over 500, 000 researchers and the widespread use of the nuclear pore complex in biology.
